# Demo: everything about one participant

After `id_mapping.ipynb`, every participant table in `final_data/` uses the same key, **`ParticipantIdentifier`** (`P0001`–`P1000`).
This notebook picks one participant and pulls their rows out of every CSV with a plain filter, no bridge tables needed.

`P0955` is used because they appear in the most tables (24 of 26). Change `PID` to look at anyone else.
Not every participant appears in every table (e.g. not everyone had a DEXA scan or opted out of SMS); an empty result is normal.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)

DATA_DIR = Path.cwd()   # run this notebook from the final_data folder
ID = "ParticipantIdentifier"
PID = "P0955"

tables = {p.stem: pd.read_csv(p, dtype=str, keep_default_na=False) for p in sorted(DATA_DIR.glob("*.csv"))}
assert "dummy_participant_master_file" in tables, f"Run from the final_data folder (cwd is {DATA_DIR})"

def rows(table: str) -> pd.DataFrame:
    """All rows of `table` belonging to PID."""
    df = tables[table]
    return df[df[ID] == PID].reset_index(drop=True)

def show(table: str, n: int = 5, transpose: bool = False) -> None:
    r = rows(table)
    print(f"{table}: {len(r)} row(s)")
    if len(r):
        display(r.head(n).T if transpose else r.head(n))

## 1. Participant profile (master file)

In [2]:
display(rows("dummy_participant_master_file").T)

,0
ParticipantIdentifier,P0955
ParticipantID,ebe52f7e-a693-4bdd-a50c-31b172ee9326
Enrollment_Date,2025-06-28
Final_Study_Status,Dropped Out
Dropout_Date,2026-01-08
Dropout_Reason,Participant withdrew consent
Age,41
Education_Level,Associate degree
Employment_Status,Unemployed
Gender_Sex,Female


The data dictionary has no participant rows; it documents the master file's columns.

In [3]:
display(tables["dummy_participant_master_data_dictionary"])

,column_name,description,data_type,example_value
0,ParticipantIdentifier,Synthetic readable participant key used across...,string,P0001
1,ParticipantID,Synthetic UUID-style participant key used in M...,string,217411dc-1a1c-4ef4-a347-b4a7ccfba050
2,Enrollment_Date,Synthetic consent/enrollment date derived from...,date (YYYY-MM-DD),2025-02-16
3,Final_Study_Status,Synthetic participant-level study outcome.,categorical (Active/Completed/Dropped Out),Completed
4,Dropout_Date,Synthetic dropout date; blank unless Final_Stu...,date (YYYY-MM-DD) or NULL,2025-08-14
5,Dropout_Reason,Synthetic clinical-team dropout reason; blank ...,categorical or NULL,Lost to follow-up
6,Age,Synthetic baseline age in years.,integer,42
7,Education_Level,Synthetic baseline education category.,categorical,Some college
8,Employment_Status,Synthetic baseline employment category.,categorical,Employed full-time
9,Gender_Sex,Synthetic baseline gender/sex category.,categorical,Female


## 2. Where the IDs came from (bridge tables)

Not needed for joining any more, but they show which NHANES respondent and on-site survey record this participant was linked to.

In [4]:
show("participant_mapping")
show("survey_participant_mapping")

participant_mapping: 1 row(s)


,ParticipantIdentifier,NHANES_SEQN
0,P0955,94708


survey_participant_mapping: 1 row(s)


,ParticipantIdentifier,RECORD_ID
0,P0955,620743


## 3. In-person exams

In [5]:
show("inperson_burden_events", n=10)

inperson_burden_events: 4 row(s)


,event_id,ParticipantIdentifier,procedure_name,location_type,start_datetime,end_datetime,duration_minutes,completion_status
0,955,P0955,Body Measures,In-Person,2018-10-12 13:22:00,2018-10-12 14:01:00,39.6,Completed
1,1955,P0955,Blood Pressure,In-Person,2018-10-11 15:28:00,2018-10-11 16:07:00,39.1,Completed
2,2551,P0955,DEXA Scan,In-Person,2018-10-09 08:58:00,2018-10-09 09:25:00,27.7,Completed
3,3292,P0955,Liver Ultrasound,In-Person,2018-10-08 09:54:00,2018-10-08 10:46:00,52.7,Completed


NHANES exam results (one row per participant, shown transposed because the tables are wide):

In [6]:
for table in ["BMX_J", "BPX_J", "LUX_J"]:
    show(table, transpose=True)

BMX_J: 1 row(s)


,0
ParticipantIdentifier,P0955
BMDSTATS,1
BMXWT,102.2
BMIWT,0
BMXRECUM,0
BMIRECUM,0
BMXHEAD,0
BMIHEAD,0
BMXHT,181.9
BMIHT,0


BPX_J: 1 row(s)


,0
ParticipantIdentifier,P0955
PEASCCT1,0
BPXCHR,0
BPAARM,1
BPACSZ,4
BPXPLS,68
BPXPULS,1
BPXPTY,1
BPXML1,150
BPXSY1,118


LUX_J: 1 row(s)


,0
ParticipantIdentifier,P0955
LUAXSTAT,1
LUARXNC,0
LUARXND,0
LUARXIN,0
LUAPNME,M
LUANMVGP,10
LUANMTGP,10
LUATECH,D
LUXSMED,4.6


In [7]:
show("DXX_J", n=1, transpose=True)  # 93 columns

DXX_J: 1 row(s)


,0
ParticipantIdentifier,P0955
DXAEXSTS,6
DXAHEBV,0
DXXHEA,0
DXXHEBMC,0
...,...
DXDTOFAT,0
DXDTOLE,0
DXDTOLI,0
DXDTOTOT,0


## 4. On-site surveys

In [8]:
show("test_data_surveys", n=10)

test_data_surveys: 11 row(s)


,ParticipantIdentifier,START_TM,FIRST_SUBMIT_TM,COMPLETE_TM,ELAPSED_TM,SURVEY_ID,SURVEY,WOMAN,AGE,WHITE,COUNTY,PHONE
0,P0955,04MAY23:17:14:00,04MAY23:17:27:00,04MAY23:17:27:00,793,37274,Stress In Childhood,1,63.58982789,1,7,2334328359
1,P0955,04MAY23:17:31:00,04MAY23:17:32:00,04MAY23:17:32:00,56,37276,Resilience,1,63.58982789,1,7,2334328359
2,P0955,04MAY23:17:11:00,04MAY23:17:12:00,04MAY23:17:12:00,90,37278,Housing Stability,1,63.58982789,1,7,2334328359
3,P0955,04MAY23:16:48:00,04MAY23:16:50:00,04MAY23:16:50:00,113,37279,Healthcare Utilization,1,63.58982789,1,7,2334328359
4,P0955,04MAY23:16:50:00,04MAY23:17:11:00,04MAY23:17:11:00,1247,37281,Financial Strain Resource Insecurity,1,63.58982789,1,7,2334328359
5,P0955,04MAY23:13:54:00,04MAY23:16:48:00,04MAY23:16:48:00,10451,37282,Dietary Screener,1,63.58982789,1,7,2334328359
6,P0955,04MAY23:17:27:00,04MAY23:17:29:00,04MAY23:17:29:00,122,37283,Depression,1,63.58982789,1,7,2334328359
7,P0955,04MAY23:17:32:00,04MAY23:18:13:00,04MAY23:18:13:00,2503,37291,AP Exit Interview,1,63.58982789,1,7,2334328359
8,P0955,04MAY23:16:48:00,04MAY23:16:48:00,04MAY23:16:48:00,6,37293,AP Sex Assigned at Birth,1,63.58982789,1,7,2334328359
9,P0955,04MAY23:16:48:00,04MAY23:16:48:00,04MAY23:16:48:00,14,37294,AP Sexual Orientation,1,63.58982789,1,7,2334328359


## 5. Remote surveys (MyDataHelps)

In [9]:
for table in ["SurveyResults", "SurveyStepResults", "SurveyStepViewed"]:
    show(table, n=3)

SurveyResults: 4 row(s)


,SurveyResultKey,SurveyKey,SurveyName,SurveyVersion,ParticipantIdentifier,SurveyTaskKey,Type,StartDate,EndDate,DevicePlatform,DeviceName,DeviceOSVersion,InsertedDate,ParticipantID,ScheduleID,ScheduleInterval,ScheduleIntervalType,ScheduleName,ScheduleCategory,UserType,User,Locale,Event
0,f928e3b2-a496-420a-b6ce-df75b5cbdfc4,de9320c2-133c-4d24-be35-6de8e7c7e678,24h Follow-up Survey,0,P0955,b8533f71-c756-4820-9cb6-110a92b1e24d,Survey,2025-06-29T08:59:19-04:00,2025-06-29T09:10:19-04:00,iOS,iPhone 14,17.5,2025-06-29T13:12:48Z,ebe52f7e-a693-4bdd-a50c-31b172ee9326,404215af-b000-465f-9346-4a8dd176583c,0,Days,24h survey schedule,Participant Burden,Participant,,,
1,34e033e2-66cc-4d10-936a-81b50f3577d2,0d639a50-52ec-4a97-8002-2cc637b8b736,7 Day Follow-up Survey,0,P0955,b63b1855-ef0f-4da0-b320-f759623f9bf9,Survey,2025-07-09T09:49:19-04:00,2025-07-09T09:52:19-04:00,iOS,iPhone 14,17.5,2025-07-09T13:53:32Z,ebe52f7e-a693-4bdd-a50c-31b172ee9326,df67abc8-bef0-445f-b90c-aa5a87a458d1,7,Days,7d survey schedule,Participant Burden,Participant,,,
2,3f90493c-0c29-4660-a3d2-9c6e292cd8f5,7000c413-9eee-4e1b-86dd-40655373dfc1,1 Month Follow-up Survey,0,P0955,0132a054-2b5c-4cf6-9a0e-1a0c3990f090,Survey,2025-07-29T08:09:19-04:00,2025-07-29T08:19:19-04:00,iOS,iPhone 14,17.5,2025-07-29T12:20:07Z,ebe52f7e-a693-4bdd-a50c-31b172ee9326,5bc4c505-4fe5-48c6-b85f-e1d011188d2d,30,Days,1m survey schedule,Participant Burden,Participant,,,


SurveyStepResults: 16 row(s)


,SurveyStepResultKey,SurveyResultKey,ParticipantIdentifier,ParticipantID,StepIdentifier,StartDate,EndDate
0,968df7d6-cc6a-480a-a8de-ec2a13cd4d58,f928e3b2-a496-420a-b6ce-df75b5cbdfc4,P0955,ebe52f7e-a693-4bdd-a50c-31b172ee9326,INTRO,2025-06-29T08:59:19-04:00,2025-06-29T08:59:42-04:00
1,bd5ad682-4a59-4568-bb28-5f97685c804d,f928e3b2-a496-420a-b6ce-df75b5cbdfc4,P0955,ebe52f7e-a693-4bdd-a50c-31b172ee9326,Q1_MOOD,2025-06-29T08:59:42-04:00,2025-06-29T08:59:56-04:00
2,358564d0-d74f-413f-b66d-74df98bb8b26,f928e3b2-a496-420a-b6ce-df75b5cbdfc4,P0955,ebe52f7e-a693-4bdd-a50c-31b172ee9326,Q2_BURDEN,2025-06-29T08:59:56-04:00,2025-06-29T09:00:18-04:00


SurveyStepViewed: 18 row(s)


,ParticipantIdentifier,Timestamp,ParticipantID,ClientLocalTimestamp,EventID,PreviousEventID,StepIdentifier,SubStepIdentifier,SubStepIndex,SurveyID,SurveySessionID,SurveyVersion,TaskID,TaskRunUUID,UtmCampaign,UtmContent,UtmMedium,UtmSource,UtmTerm,Fbclid,Gclid,Ndclid
0,P0955,2025-06-29T12:59:19Z,ebe52f7e-a693-4bdd-a50c-31b172ee9326,2025-06-29T08:59:19-04:00,1b15feba-8e00-49a9-b8b9-eaf03f555a02,,INTRO,,,de9320c2-133c-4d24-be35-6de8e7c7e678,0437c003-eff4-49f8-a479-6ae86df2040b,0,b8533f71-c756-4820-9cb6-110a92b1e24d,fda61e8a-fae7-4747-9bec-0103198b782d,,,,,,,,
1,P0955,2025-06-29T13:00:28Z,ebe52f7e-a693-4bdd-a50c-31b172ee9326,2025-06-29T09:00:28-04:00,ae548506-fe1c-4f62-ba56-47ee39fca712,1b15feba-8e00-49a9-b8b9-eaf03f555a02,Q1_MOOD,,,de9320c2-133c-4d24-be35-6de8e7c7e678,0437c003-eff4-49f8-a479-6ae86df2040b,0,b8533f71-c756-4820-9cb6-110a92b1e24d,fda61e8a-fae7-4747-9bec-0103198b782d,,,,,,,,
2,P0955,2025-06-29T13:01:14Z,ebe52f7e-a693-4bdd-a50c-31b172ee9326,2025-06-29T09:01:14-04:00,7f25304e-e115-4702-aad3-127a2e059ca6,ae548506-fe1c-4f62-ba56-47ee39fca712,Q2_BURDEN,,,de9320c2-133c-4d24-be35-6de8e7c7e678,0437c003-eff4-49f8-a479-6ae86df2040b,0,b8533f71-c756-4820-9cb6-110a92b1e24d,fda61e8a-fae7-4747-9bec-0103198b782d,,,,,,,,


## 6. Communication (MyDataHelps)

In [10]:
communication = [
    "NotificationSent",
    "EmailSend", "EmailDelivery", "EmailOpen", "EmailClick", "EmailBounce", "EmailComplaint",
    "SmsSend", "SmsDelivery", "SmsFailure", "SmsLinkClicked", "SmsOptOut",
    "PushNotificationSent", "PushNotificationOpened",
]
for table in communication:
    show(table, n=3)

NotificationSent: 9 row(s)


,ParticipantID,ParticipantIdentifier,Timestamp,ContentVersion,NotificationIdentifier,NotificationType,Recipients,StatusCode,TargetCustomField,RecipientOverride
0,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,2025-06-28T00:49:19Z,2,24H_SURVEY_REMINDER_1,SMS,p0955@example.org,Succeeded,,
1,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,2025-07-05T00:49:19Z,5,7D_SURVEY_REMINDER_1,Email,p0955@example.org,Succeeded,,
2,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,2025-07-06T00:49:19Z,2,7D_SURVEY_REMINDER_2,Email,p0955@example.org,Succeeded,,


EmailSend: 4 row(s)


,ParticipantID,ParticipantIdentifier,MessageID,EmailType,TemplateName,ToEmailAddress,SenderEmailAddress,ContentVersion,Subject
0,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,7e04232e-cb64-4ab1-a937-6dd92e2ecd20,Reminder,7D_SURVEY_REMINDER_1,p0955@example.org,study-team@example.org,4,Survey reminder
1,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,1ebdc86a-7dcf-4acd-b830-a50c4bba43f9,Invitation,7D_SURVEY_REMINDER_2,p0955@example.org,study-team@example.org,1,Survey reminder
2,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,6a9d8c73-0593-4c75-8ee5-794efdaea022,Reminder,7D_SURVEY_REMINDER_3,p0955@example.org,study-team@example.org,5,Survey reminder


EmailDelivery: 4 row(s)


,ParticipantID,ParticipantIdentifier,MessageID,EmailType,TemplateName,ToEmailAddress,SenderEmailAddress,ContentVersion,Subject,DeliveryRecipient,DeliveryTimestamp
0,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,7e04232e-cb64-4ab1-a937-6dd92e2ecd20,Reminder,7D_SURVEY_REMINDER_1,p0955@example.org,study-team@example.org,4,Survey reminder,p0955@example.org,2025-07-05T00:49:49Z
1,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,1ebdc86a-7dcf-4acd-b830-a50c4bba43f9,Reminder,7D_SURVEY_REMINDER_2,p0955@example.org,study-team@example.org,2,Survey reminder,p0955@example.org,2025-07-06T00:49:49Z
2,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,6a9d8c73-0593-4c75-8ee5-794efdaea022,Reminder,7D_SURVEY_REMINDER_3,p0955@example.org,study-team@example.org,1,Survey reminder,p0955@example.org,2025-07-07T00:49:49Z


... (15 outputs left out)

,ParticipantID,ParticipantIdentifier,MessageID,PhoneNumber,MessageType,OriginalLink,ShortenedLink,Timestamp
0,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,fec34504-f153-43e1-9451-2a6817d9daf9,+15550000955,SurveyLink,https://example.org/survey,https://mdh.li/abc123,2025-09-26T09:39:19Z


SmsOptOut: 0 row(s)
PushNotificationSent: 2 row(s)


,ParticipantID,ParticipantIdentifier,Timestamp,PushNotificationIdentifier,StatusCode
0,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,2025-12-16T01:49:19Z,e46bcdc7-e845-4145-80e4-6b7973da997c,Succeeded
1,ebe52f7e-a693-4bdd-a50c-31b172ee9326,P0955,2025-12-17T01:49:19Z,e46bcdc7-e845-4145-80e4-6b7973da997c,Succeeded


PushNotificationOpened: 0 row(s)


## 7. Coverage summary

How many rows `PID` has in each table that carries `ParticipantIdentifier`.

In [11]:
coverage = pd.DataFrame(
    [(name, len(rows(name))) for name, df in tables.items() if ID in df.columns],
    columns=["table", "rows"],
).sort_values("rows", ascending=False, ignore_index=True)

print(f"{PID} appears in {(coverage['rows'] > 0).sum()} of {len(coverage)} tables")
coverage

P0955 appears in 24 of 26 tables


,table,rows
0,SurveyStepViewed,18
1,SurveyStepResults,16
2,test_data_surveys,11
3,NotificationSent,9
4,inperson_burden_events,4
5,SurveyResults,4
6,EmailDelivery,4
7,EmailOpen,4
8,EmailSend,4
9,SmsSend,3


## 8. Combining sources

Because everything shares the key, sources can be combined directly. Example: time spent so far per activity type.
Each table is aggregated per participant **before** combining, so rows from different tables are never multiplied together.

In [12]:
inperson = rows("inperson_burden_events")
inperson = inperson[inperson["completion_status"] == "Completed"]
onsite = rows("test_data_surveys")

time_spent = pd.DataFrame({
    "source": ["In-person exams (completed)", "On-site surveys"],
    "items": [len(inperson), len(onsite)],
    "minutes": [
        inperson["duration_minutes"].astype(float).sum(),
        onsite["ELAPSED_TM"].astype(float).sum() / 60,   # ELAPSED_TM is in seconds
    ],
}).round(1)
time_spent

,source,items,minutes
0,In-person exams (completed),4,159.1
1,On-site surveys,11,258.4
